# Comparing results

The code within this workbook reads nodal displacement from *.sin files, and plots the data.

For installation of dependencies, see [DNV SIF-IO pypi page](https://pypi.org/project/dnv-sifio/)



In [2]:
import dnv.net.runtime
from dnv.sesam.sifapi.core import ISifData
from dnv.sesam.sifapi.io import SesamDataFactory
import os
import numpy as np
import pandas as pd
from qats import TsDB, TimeSeries
import glob


## Define functions for getting data from sin file

Data is returned as numpy arrays

In [ ]:


def get_series_for_element_reverse(sin_file,internal_node_nos):
    """
    Reads nodal displacement data (RVNODDIS) of all result cases and given FE node in the given SIN file
    
    Note: switching the order in the double for-loop is significantly slower
    
    """

    root, file = os.path.split(sin_file) 
    run_root = os.getcwd()
    os.chdir(root)
    if not isinstance(internal_node_nos, list):
        internal_node_nos = [internal_node_nos]

    with SesamDataFactory.CreateReader('.', file) as adapter:
        adapter.CreateModel()
        lc_no = adapter.GetCount('RDRESREF')
        data = np.empty((len(internal_node_nos), 6, lc_no))
        
        for j, node_no in enumerate(internal_node_nos):
            for i in range(lc_no):
                res = adapter.ReadExt('RVNODDIS', [i+1, node_no])
               
                res = [e for e in res]
                data[j,:,i] = res[6:]

    os.chdir(run_root)
    return data


## Find files to compare

In [7]:
resfiles = glob.glob('../Truncated/**/*_R3.sin',recursive=True)

Extract nodal displacements for selected nodes, from all selected sin files

In [ ]:

nodes = [9653, 5017, 6759, 9811, 7973, 1181]
datas = {}
for resfile in resfiles:
    if "Lowres" in resfile or "stress" in resfile:
        continue

    hierachy = resfile.split('\\')
    dlc = hierachy[-2].split('-')[0]
    meth = hierachy[-3]
    data =  get_series_for_element_reverse(resfile, nodes)
    if not meth in datas.keys():
        datas[meth] = {}
    datas[meth][dlc] = data

Export the data into QATS files. 
See [QATS github page](https://github.com/dnvgl/qats) for more info

In [14]:
os.chdir(r'E:\Whitpaper-TD-Methods\Analyses\comparison')
db = TsDB()


for meth in datas.keys():
    for dlc in datas[meth].keys():
        data = datas[meth][dlc]
        db = TsDB()
        t = np.arange(0,data.shape[2])
        for i, node in enumerate(nodes):
            for j, response_type in enumerate(['X','Y','Z', 'Rx','Ry','Rz']):
                ts = TimeSeries(f'{node}-{response_type}',t,data[i,j,:])
                db.add(ts)
        db.export(f'response-{meth}-{dlc}.ts')



Exported 36 records to file 'response-DirectComp-DLC2.ts'.
Exported 36 records to file 'response-DirectComp-DLC3.ts'.
Exported 36 records to file 'response-DirectComp-DLC4.ts'.
Exported 36 records to file 'response-DirectComp-DLC5.ts'.
Exported 36 records to file 'response-LoadReconstruction-DLC2.ts'.
Exported 36 records to file 'response-LoadReconstruction-DLC3.ts'.
Exported 36 records to file 'response-LoadReconstruction-DLC4.ts'.
Exported 36 records to file 'response-LoadReconstruction-DLC5.ts'.
Exported 36 records to file 'response-ResponsReconstruction-DLC2.ts'.
Exported 36 records to file 'response-ResponsReconstruction-DLC3.ts'.


The exported data can be looked at using the QATS GUI. Alternatively, the data can be investigated using other python tools